In [1]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

Cloning into 'temporalgnn-nids'...
remote: Enumerating objects: 814, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 814 (delta 6), reused 8 (delta 2), pack-reused 791 (from 1)
Receiving objects: 100% (814/814), 5.27 MiB | 10.64 MiB/s, done.
Resolving deltas: 100% (266/266), done.
Mounted at /content/drive


In [2]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.7 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import copy
import random
import json
import time

import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader


In [4]:
from utils.datasets   import NF_IDS_Dataset
from utils.models     import E_GraphSAGE_Entropy
from utils.metrics    import calculate_metrics_gnn
from utils.training   import train_epoch, evaluate, find_optimal_threshold, set_seed, run_multiple_seeds
from utils.experiment import ExperimentManager, EarlyStopping, NumpyEncoder
from utils.visualization import save_plots

# Auxiliary

In [5]:
ROOT_PATH = "./dataset_processed_entropy"

In [6]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [7]:
ROOT_DIR = "./results_earlystopping_entropy"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")


# Main

## Seeds

In [8]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

#NODE_DIM = 16
EDGE_DIM = 32   # 7 dst_port + 5 protocol + 20 numeric
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [9]:
model_config = {
    "model_name": None,
    "type": "E_GraphSAGE_Entropy",
    "model_params": {
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [10]:
EXPERIMENT_NAME="EGraphSAGE_Entropy_BiasOn"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [12]:
run_multiple_seeds(
    model_class=E_GraphSAGE_Entropy,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: EGraphSAGE_Entropy_BiasOn
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=EGraphSAGE_Entropy_BiasOn_seed42_20260419_131730

EGraphSAGE_Entropy_BiasOn_seed42
   Ep 1 | Loss: 0.2193 | Val Loss: 0.2470 | Val AUC-PR: 0.0780 (*)
   Ep 3 | Loss: 0.2141 | Val Loss: 0.2433 | Val AUC-PR: 0.0898 (*)
   Ep 5 | Loss: 0.2008 | Val Loss: 0.2165 | Val AUC-PR: 0.1159 (*)
   Ep 6 | Loss: 0.1916 | Val Loss: 0.2289 | Val AUC-PR: 0.1446 (*)
   Ep 7 | Loss: 0.1953 | Val Loss: 0.2371 | Val AUC-PR: 0.2492 (*)
   Ep 10 | Loss: 0.1809 | Val Loss: 0.2068 | Val AUC-PR: 0.2366 
   Ep 12 | Loss: 0.1797 | Val Loss: 0.2024 | Val AUC-PR: 0.2635 (*)
   Ep 14 | Loss: 0.1751 | Val Loss: 0.2007 | Val AUC-PR: 0.2910 (*)
   Ep 15 | Loss: 0.1803 | Val Loss: 0.2034 | Val AUC-PR: 0.2997 (*)
   Ep 16 | Loss: 0.1747 | Val Loss: 0.1981 | Val AUC-PR: 0.3099 (*)
   Ep 17 | Loss: 0.1690 | Val Loss: 0.1982 | Val AUC-PR: 0.3146 (*)
   